# Exploratory Data Analysis & Statistical Sensor Audit
In this notebook, we load the raw UCI SECOM semiconductor manufacturing dataset, split the production batches chronologically into Phase I (In-Control Calibration) and Phase II (Online Monitoring), and perform a strict statistical audit to filter out corrupted, constant, and redundant sensors without data leakage.

In [1]:
# Imports for data manipulation
import pandas as pd
import numpy as np

## 1. Loading and Inspecting the SECOM Dataset
First, we read the raw telemetry file (`uci-secom.csv`), rename the 590 sensor columns, sort all production batches chronologically by timestamp, and encode the quality outcome as a binary target (`0` = In-Control / Pass, `1` = Defect / Fail).

In [7]:
# Load the SECOM dataset (semicolon-separated with comma decimal format)
df = pd.read_csv('../data/uci-secom.csv', sep=';', decimal=',')

# Rename sensor columns from '0'..'589' to 'Sensor_000'..'Sensor_589'
sensor_cols = [f'Sensor_{int(c):03d}' for c in df.columns if c not in ['Time', 'Pass/Fail']]
rename_dict = {str(i): f'Sensor_{i:03d}' for i in range(590)}
df = df.rename(columns=rename_dict)

# Parse timestamps and sort batches chronologically
df['Time'] = pd.to_datetime(df['Time'], format='mixed', dayfirst=False)
df = df.sort_values('Time').reset_index(drop=True).copy()

# Binary quality target: 0 = In-Control (Pass = -1), 1 = Defective (Fail = 1)
df['Defect'] = (df['Pass/Fail'] == 1).astype(int)
df = df.drop(columns=['Pass/Fail'])

In [8]:
# Check dataset dimensions, time span, and overall defect rate
print(f"Dimensions of the dataset: {df.shape[0]} production batches x {len(sensor_cols)} sensors")
print(f"Time range: {df['Time'].min()} to {df['Time'].max()}")
print(f"Across all production batches, {df['Defect'].value_counts()[1]} are defective ({round(df['Defect'].mean() * 100, 2)}% defect rate).")

Dimensions of the dataset: 1567 production batches x 590 sensors
Time range: 2008-01-08 02:02:00 to 2008-12-10 18:47:00
Across all production batches, 104 are defective (6.64% defect rate).


## 2. Phase I (Calibration) vs. Phase II (Online Monitoring) Split
In Statistical Process Control (SPC), baseline parameters and control limits must never be estimated using the entire dataset at once. We split our timeline into two stages:
* **Phase I (Batches 1–800)**: Historical calibration period. We use only the In-Control (`Defect == 0`) batches from Phase I to learn the normal process medians, variances, and sensor correlations.
* **Phase II (Batches 801–1,567):** Future production batches used to evaluate online monitoring charts.

In [4]:
# Split batches into Phase I (first 800 batches) and Phase II (remaining 767 batches)
df['Phase'] = np.where(df.index < 800, 'Phase I', 'Phase II')

# Reference In-Control baseline from Phase I
df_phase1_ic = df[(df['Phase'] == 'Phase I') & (df['Defect'] == 0)]

print(f"Total batches in Phase I: {(df['Phase'] == 'Phase I').sum()} ({len(df_phase1_ic)} of them In-Control)")
print(f"Total batches in Phase II: {(df['Phase'] == 'Phase II').sum()} ({len(df[(df['Phase'] == 'Phase II') & (df['Defect'] == 1)])} of them defective)")

Total batches in Phase I: 800 (728 of them In-Control)
Total batches in Phase II: 767 (32 of them defective)


## 3. Statistical Sensor Audit
We audit all 590 sensors exclusively on `df_phase1_ic` to prevent data leakage into Phase II:
1. **Missing Data Filter (`> 40% NaN`):** Remove sensors with unreliable or sparse sampling.
2. **Zero-Variance Filter (`std < 1e-6`):** Remove constant or turned-off sensors that provide zero statistical information.
3. **Extreme Collinearity Filter (`|r| > 0.98`):** Remove redundant sensor channels so the covariance matrix remains invertible for Multivariate SPC (Hotelling's $T^2$).

In [ ]:
# 1. Remove sensors with more than 40% missing values in Phase I In-Control
na_p1 = df_phase1_ic[sensor_cols].isna().mean()
high_na_sensors = na_p1[na_p1 > 0.40].index.to_list()
remaining_1 = [s for s in sensor_cols if s not in high_na_sensors]

# 2. Remove constant sensors (standard deviation < 1e-6 in Phase I In-Control)
std_p1 = df_phase1_ic[remaining_1].std()
zero_var = std_p1[std_p1 < 1e-6].index.to_list()
remaining_2 = [s for s in remaining_1 if s not in zero_var]

# 3. Impute Phase I medians temporarily to evaluate pairwise sensor collinearity
median_p1 = df_phase1_ic[remaining_2].median()
X_p1_imp = df_phase1_ic[remaining_2].fillna(median_p1)

# Remove sensors with extreme correlation (|r| > 0.98 in Phase I)
corr_matrix = X_p1_imp.corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = [s for s in upper_tri if any(upper_tri[s] > 0.98)]

valid_sensors = [s for s in remaining_2 if s not in high_corr]

# Build summary table of the sensor audit
audit_tab = pd.DataFrame([
    {"Step": "1. Initial Raw Sensors", "Sensors_Removed": 0, "Sensors_Remaining": len(sensor_cols), "Reason": "Raw SECOM telemetry input"},
    {"Step": "2. Missing Data Filter (>40% NaN)", "Sensors_Removed": len(high_na_sensors), "Sensors_Remaining": len(remaining_1), "Reason": "Unreliable / sparse sampling in Phase I"},
    {"Step": "3. Zero-Variance Filter (std < 1e-6)", "Sensors_Removed": len(zero_var), "Sensors_Remaining": len(remaining_2), "Reason": "Constant / turned-off sensors"},
    {"Step": "4. Extreme Collinearity Filter (|r| > 0.98)", "Sensors_Removed": len(high_corr), "Sensors_Remaining": len(valid_sensors), "Reason": "Redundant channels (prevents singular covariance matrix)"}
])

audit_tab 

,Step,Sensors_Removed,Sensors_Remaining,Reason
0,1. Initial Raw Sensors,0,590,Raw SECOM telemetry input
1,2. Missing Data Filter (>40% NaN),28,562,Unreliable / sparse sampling in Phase I
2,3. Zero-Variance Filter (std < 1e-6),122,440,Constant / turned-off sensors
3,4. Extreme Collinearity Filter (|r| > 0.98),126,314,Redundant channels (prevents singular covarian...


## 4. Median Imputation and Saving Clean Telemetry
Finally, we impute any remaining missing values across the 314 retained sensors using exclusively the Phase I In-Control medians (which are robust against outliers and prevent data leakage from Phase II) and export both the cleaned dataset (`data/secom_cleaned.csv`) and the Markdown summary table (`reports/sensor_auditoria_summary.md`).

In [ ]:
# Impute remaining missing values using only Phase I In-Control medians
medians_p1 = df_phase1_ic[valid_sensors].median()
df_clean = df[['Time', 'Defect', 'Phase'] + valid_sensors].copy()
df_clean[valid_sensors] = df_clean[valid_sensors].fillna(medians_p1)

# Save the cleaned dataset and the Markdown summary table
df_clean.to_csv('../data/secom_cleaned.csv', index=False)
with open('../reports/sensor_auditoria_summary.md', 'w', encoding='utf-8') as f:
    f.write(audit_tab.to_markdown(index=False))

Clean dataset saved to '../data/secom_cleaned.csv' with 314 valid sensors and 0 missing values.
